# word2vec

Word2Vec 是一种词嵌入（Word Embedding）方法，它可以计算每个单词在其给定语料库环境下的分布式词向量（Distributed Representation，亦直接被称为词向量）。词向量表示可以在一定程度上刻画每个单词的语义。

## 简单用法

### 读取语料

- 语料可以存储在内存中，格式为`[[word1,word2,word3...],[word1,word2,word3...],...]`，列表中每一个子列表为分完词的一篇文档
- `class gensim.models.word2vec.LineSentence(source, max_sentence_length=10000, limit=None)`
  source为可读文件路径，文件每一行代表一篇文档，文档是已经经过分词，每个词由空格分隔。max_sentence_length为文章的最大长度，limit为读取前多少篇文档(即前多少行)
- `class gensim.models.word2vec.PathLineSentences (source, max_sentence_length = 10000, limit = None )` 与LineSentence类似，不过这里传入的是根目录，目录下有多个可读文件，文件格式需要和与LineSentence所需文件格式类似，此函数可处理根目录下所有的文件。

In [11]:
import jieba
from gensim.models import word2vec

#### 1.内存方式

In [ ]:
# 加载自定义词典
jieba.load_userdict('data/phone_dict.txt')

# 停用词
filepath = 'data/stopwords.txt'
stopwords = [line.strip() for line in open(filepath, 'r', encoding='utf-8')]

In [6]:
# 读取文件
def rf2wl(filepath):
    cut_list = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            line = line.strip()
            seg_list = jieba.cut(line)
            seg_list = [word for word in seg_list if word not in stopwords and word != ' ']
            cut_list.append(seg_list)
    return cut_list

In [7]:
# 未分词语料
filepath = 'data/mb.txt'
cut_list = rf2wl(filepath)

In [8]:
print(cut_list[:10])

[['Apple', 'iPhone', 'Plus', 'A1864', '64GB', '深空', '灰色', '移动', '联通', '电信', '4G', '手机'], ['Apple', 'iPhone', 'Plus', 'A1661', '128G', '黑色', '移动', '联通', '电信', '4G', '手机'], ['OPPO', 'KTx', '双模', '5G', '4800', '万四摄', '5000mAh', '长', '续航', '90Hz', '电', '竞屏', '蓝影', '6GB', '128GB', '30W', '闪充', '全', '网通', '游戏', '智能手机'], ['一加', '手机', 'OnePlus', '8T', '5G', '旗舰', '120Hz', '柔性', '直屏', '65W', '闪充', '高通', '骁龙', '865', '超强', '四摄', '12GB', '256GB', '青域', '拍照', '游戏', '手机'], ['小米', '红米', 'Plus', '全面', '屏', '拍照', '手机', '全', '网通', '版', '3GB', '32GB', '金色', '移动', '联通', '电信', '4G', '手机', '双卡', '双待'], ['Apple', 'iPhone', 'A1660', '128G', '黑色', '移动', '联通', '电信', '4G', '手机'], ['Apple', 'iPhone', 'X', 'A1865', '64GB', '深空', '灰色', '移动', '联通', '电信', '4G', '手机'], ['小米', '红米', 'Note5A', '移动', '4G', '版全', '网通', '4GB', '64GB', '铂', '银灰', '移动', '联通', '电信', '4G', '手机', '双卡', '双待', '拍照', '手机'], ['荣耀', 'V10', '全', '网通', '标配', '版', '4GB', '64GB', '幻', '夜黑', '移动', '联通', '电信', '4G', '全', '画屏', '手机', '双卡', '双待'], ['Redmi'

#### 2.文件方式


In [12]:
file_path = 'data/mb_train.txt'
sentences = word2vec.LineSentence(file_path,max_sentence_length=10000,limit=None)

In [13]:
sentences

In [14]:
for doument in sentences:
    print(doument)
    break

['Apple', 'iPhone', 'Plus', 'A1864', '64GB', '深空灰', '色', '移动', '联通', '电信', '4G', '手机']


### 训练word2vec语义向量

---

```python
class gensim.models.word2vec.Word2Vec(sentences=None, size=100, alpha=0.025, window=5, min_count=5,max_vocab_size=None, sample=1e-3, seed=1, workers=3, min_alpha=0.0001,sg=0, hs=0, negative=5, ns_exponent=0.75, cbow_mean=1, hashfxn=hash, iter=5, null_word=0,trim_rule=None, sorted_vocab=1, batch_words=MAX_WORDS_IN_BATCH, compute_loss=False, callbacks=(),max_final_vocab=None)
```

- `sentence(iterable of iterables)`:输入语料，与我们上面生成的一致
- `SG(INT {1, 0})`-定义的训练算法。如果是1，则使用skip-gram; 否则，使用CBOW。
- `hs`: 是否采用基于Hierarchical Softmax的模型。参数为1表示使用，0表示不使用
- `size(int)` - 特征向量的维数。 
> 从 4.0 开始 size 已更名为 vector_size
- `window(int)` - 句子中当前词和预测词之间的最大距离。
- `min_count(int)` - 忽略总频率低于此值的所有单词。

In [20]:
model = word2vec.Word2Vec(sentences,sg=1,hs=1,window=5,min_count=1,vector_size=200)


### 保存模型

---

```python
model.save("file_name")
```

In [21]:
model.save('models/mb_word2vec.model')

### 加载模型

In [22]:
model = word2vec.Word2Vec.load('models/mb_word2vec.model')

In [26]:
# 获取词表
print(model.wv.index_to_key )

['5G', '快充', '手机', '屏', '12GB', '万', '8GB', '256GB', '三摄', 'Pro', '5000', '旗舰', '128GB', '6GB', '4G', '120Hz', '全网通', '闪充', '电池', '4800万', '超级', '像素', '亿', '移动', '三星', '6400', '主摄', '荣耀', '华为', '双卡双待', 'Galaxy', '小米', '电信', '联通', '四摄', 'OPPO', 'AI', 'Redmi', '一加', '大', 'vivo', '18W', '直屏', '智能手机', '33W', '66W', '90Hz', '120HzOLED', '黑', '双摄', '超', '游戏手机', 'iPhone', 'Apple', '骁龙', '拍照手机', '影像', '5000mAh', '蓝', '曲面', '120W', '红米', '白', '100W', '67W', 'W', '22.5', '120Hz2K', '卡', '柔性', 'Ultra', '40W', 'Plus', '银', '65W', '25W', 'Nova', '12', '万超', '视网膜', '智慧', '万徕', '30W', '续航', '性能', '120HzAMOLED', '4GB', '长', '120HzLTPO', '80W', '万主摄', '黑色', '15', 'Note13', 'IMX800', '15W', '金', '绿', '64GB', '11', '感光', '14', 'Ace', '皇', '4000mAh', '灰', 'Z', '哈苏', '刷屏', '高', '1TB', '小金刚', 'Turbo', '安卓机', '青', '天玑', '迷雾', '60', '90HzSuperAMOLED', '全能', '紫', 'Max', 'K70', '曜', 'iQOO', 'Mate', '35W', 'Gen3', 'IMX890', '6000mAh', '4800mAh', '超清', 'Find', '120HzLTPO3.0', '翡翠绿', 'Reno12', '27W', '60HzLiquid',

In [29]:
# 获取单词word2vec值
model.wv['5G']


array([-2.79758256e-02,  3.84375863e-02, -4.33319844e-02,  5.53629436e-02,
       -4.69723754e-02, -1.91068664e-01, -8.01363140e-02,  2.76943237e-01,
        5.02714552e-02,  1.07267126e-01, -1.00890577e-01, -2.20643252e-01,
        1.02390079e-02,  4.03529666e-02, -3.68219167e-02, -1.60765454e-01,
       -4.81335856e-02, -1.92315970e-03,  4.66260239e-02, -1.68372214e-01,
        1.28974602e-01, -1.92803741e-01,  4.64577973e-02,  7.93410931e-03,
        6.41631335e-02, -1.32953286e-01, -1.76827058e-01,  5.12427613e-02,
       -9.88741890e-02, -5.17600812e-02,  2.28116773e-02, -9.20276195e-02,
        2.99737956e-02, -6.79747760e-02,  4.25686203e-02, -2.63421182e-02,
        2.80490257e-02, -9.98967886e-02,  1.08387649e-01, -2.83328574e-02,
       -7.23869428e-02,  1.86873674e-01, -8.26429874e-02, -1.14741444e-03,
        4.63025384e-02,  2.88017504e-02, -1.86873294e-04,  3.49326385e-03,
        1.29165679e-01,  4.88445275e-02, -9.29273292e-02, -2.76044719e-02,
        9.29425508e-02,  

> word2vec值在gensim4.0.0之后的版本中移除了`model['5G']`，需要使用`model.wv['5G']`获取。

In [33]:
# 计算单词语义相似度
print(model.wv.similarity('5G', '4G'))
print(model.wv.similarity('手机', '4G'))
print(model.wv.similarity('手机', '8GB'))

0.35398522
0.9759427
0.830906
